In [1]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcraft.trainer import ThermodynamicTrainer
from imolcraft.trainer.dmff_utils import parser_dmffyaml
from imolcraft.trainer.loss import loss_thermodynamicperturbation
from functools import partial

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'merged_supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 1.2,
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_steps': 1,
  'anneal_Tmax': 400.0,
  'anneal_totalsteps': 0,
  'relax_steps': 20000,
  'prod_steps': 100000,
  'nstxout': 1000,
  'neff': 30,
  'dispcorr': True,
  'anneal_totaltime': 0},
 'targets': {'density_gcm3': {'weight': 1.0, 'gt': 1.568},
  'La_A': {'weight': 1.0, 'gt': 36.82},
  'Lc_A': {'weight': 1.0, 'gt': 40.2152},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}}}

In [3]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="wrightfactor")

In [4]:
trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="merged_supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charge", "NonbondedForce/epsilon", "NonbondedForce/sigma"],
    label="2rdf_1adf_density",
)

In [5]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1153, 0.1467, 0.1538, 0.1097, 0.1612, 0.1453, 0.1719], dtype=float64),
  'k': Array([746040.672, 247575.648, 194572.736, 314569.856, 335431.28 ,
         571559.504, 237098.912], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([3.11680898, 1.94970731, 1.90956473, 1.91637152, 1.87762521,
         2.08881005, 1.85947379, 1.69183491, 2.09526777, 1.86785137],      dtype=float64),
  'k': Array([ 612.671488,  554.597568,  409.019472,  391.756288,  326.01728 ,
          530.556304, 1089.488496,  567.551232, 1072.7776  , 1026.703392],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([3.14159265, 3.14159265, 0.        , 0.        , 0.        ,
         2.16387219, 2.27151958], dtype=float64),
  'proper_k': Array([ 0.        ,  0.        ,  0.65084444,  0.65084444,  0.50208   ,
         45.65444   , 35.24682804], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dt

In [6]:
from imolcraft.trainer.dmff_utils import neutralize

def grad_modify(grads):
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[10].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[11].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[12].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[13].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[14].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[15].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[16].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[17].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[18].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[19].set(0.0)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list, target_lists=[[0,1,2,3,4,5,6,7,8,9]], target_charges=[0.0])

    return ffparams

In [7]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [8]:
trainer.setup()

Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Potential Energy (kJ/mole)","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,-117431.01098277365,1.499296044310578,0,--
2000,-117034.19488169943,1.4998288221175862,184,0:55
3000,-117200.61114146505,1.4921076641480964,183,0:55
4000,-117172.23522593771,1.4874978925059383,183,0:54
5000,-117242.0671351174,1.4960069890868382,182,0:54
6000,-117285.25591685568,1.495741442794954,182,0:54
7000,-117278.4519007424,1.4973628739369145,182,0:53
8000,-117206.45165660177,1.4902974247463916,182,0:53
9000,-117182.17419078146,1.4972099369729612,182,0:52
10000,-116968.59228160177,1.5025102920088973,182,0:52
11000,-117059.15038707052,1.4929824923050372,182,0:51
12000,-117100.24200084005,1.496701224867803,182,0:51
13000,-117089.4866297463,1.4963852960747686,182,0:50
14000,-116894.77672984396,1.488423226021533,182,0:50
15000,-117101.87273814474,1.4949386143842587,182,0:49
16000,-117141.303707

/Users/srak/miniconda3/envs/imc/lib/python3.11/site-packages/MDAnalysis/coordinates/XDR.py:261: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(
100%|██████████| 100/100 [00:02<00:00, 38.76it/s]


In [9]:
neff_prev = trainer.neff
print("Initial neff:", neff_prev)

Initial neff: [30]


In [10]:
trainer.neff = [50]

In [ ]:
trainer.fit(10, 2)

Pair list: 100%|██████████| 100/100 [00:30<00:00,  3.29it/s]


Init Energy calc: Memory Usage: 12568.67 MB
Energy calc: 0.79 sec. Memory Usage: 12070.41 MB
Get weight, Memory Usage: 6155.08 MB
Finish loss calc Memory Usage: 6207.48 MB
grad obtained.... Memory Usage: 8346.67 MB
grad modify.... Memory Usage: 5781.52 MB
update finished.... Memory Usage: 429.78 MB
Effective sample sizes:
  sample_0: 58
  Total: 58.88020671638841
Epoch 0, Loss: 1.06537671191451, Memory Usage: 498.47 MB
Loss:  1.06537671191451
Best Loss:  1.06537671191451 at epoch  0
Epoch 0 completed in 101.73 seconds.
----
Init Energy calc: Memory Usage: 500.19 MB
Energy calc: 0.01 sec. Memory Usage: 544.58 MB
Get weight, Memory Usage: 6132.09 MB
Finish loss calc Memory Usage: 6149.97 MB
grad obtained.... Memory Usage: 6293.30 MB
grad modify.... Memory Usage: 14173.84 MB
update finished.... Memory Usage: 6792.12 MB
Effective sample sizes:
  sample_0: 56
  Total: 57.688653634662174
Loss:  1.0563778672734268
Best Loss:  1.0563778672734268 at epoch  1
Epoch 1 completed in 57.54 seconds.


100%|██████████| 100/100 [00:02<00:00, 39.71it/s]


Epoch 2, Loss: 1.0474499166699025, Memory Usage: 1661.12 MB
Loss:  1.0474499166699025
Best Loss:  1.0474499166699025 at epoch  2
Epoch 2 completed in 139.13 seconds.
----


Pair list: 100%|██████████| 100/100 [00:34<00:00,  2.93it/s]


Init Energy calc: Memory Usage: 12587.73 MB
Energy calc: 0.68 sec. Memory Usage: 9873.95 MB
Get weight, Memory Usage: 5639.38 MB
Finish loss calc Memory Usage: 5660.28 MB
grad obtained.... Memory Usage: 6055.09 MB
grad modify.... Memory Usage: 13021.88 MB
update finished.... Memory Usage: 8837.39 MB
Effective sample sizes:
  sample_0: 53
  Total: 54.26481533493078
Loss:  1.0676005307858272
Best Loss:  1.0474499166699025 at epoch  2
Epoch 3 completed in 108.85 seconds.
----
Init Energy calc: Memory Usage: 5653.62 MB
Energy calc: 0.01 sec. Memory Usage: 5752.12 MB
Get weight, Memory Usage: 5742.75 MB
Finish loss calc Memory Usage: 5769.03 MB
grad obtained.... Memory Usage: 5887.56 MB
grad modify.... Memory Usage: 15617.45 MB
update finished.... Memory Usage: 11942.38 MB
Effective sample sizes:
  sample_0: 62
  Total: 63.48390225782241
Epoch 4, Loss: 1.052578426740211, Memory Usage: 6893.08 MB
Loss:  1.052578426740211
Best Loss:  1.0474499166699025 at epoch  2
Epoch 4 completed in 60.21 s

In [ ]:
trainer.get_loss_gradients()

In [ ]:
trainer.neff = neff_prev

In [ ]:
trainer.fit(490, 2)

In [ ]:
trainer.fit(40, 2)

In [ ]:
# trainer.from_checkpoint(trainer_checkpoint="train_state.pkl",
#                         ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
#                         nums_ffxml=[2,1,1],
#                         pdbfile="merged_supercell_bonds.pdb",
#                         initial_ffxml="xmlfiles/epoch-4.xml",
#                         loss_fn=lossfn,
#                         opt_fftypes=["NonbondedForce/charges"],
#                         )